# Notebook 04

O Notebook 02 identificou autores com as maiores médias de avaliação. Contudo, a média isoladamente oculta um aspecto fundamental para sistemas de recomendação: a consistência. Dois autores podem compartilhar a mesma média geral, mas enquanto um entrega qualidade estável em toda sua produção, o outro alterna entre obras excelentes e mediocres.

Este notebook investiga essa variação utilizando o desvio padrão das notas como métrica central, aplicada a todos os autores com três ou mais obras no catálogo. A análise busca responder: recomendar um autor é o suficiente, ou é preciso recomendar a obra certa do autor certo?

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_parquet('datasets/books.parquet')

# Apenas autores com 3+ obras — mínimo necessário para calcular desvio padrão com representatividade
contagem = df['author'].value_counts()
autores_validos = contagem[contagem >= 3].index
df_autores = df[df['author'].isin(autores_validos)]

stats = df_autores.groupby('author').agg(
    qtd_livros=('title', 'count'),
    media_nota=('rating', 'mean'),
    desvio_nota=('rating', 'std'),
    media_paginas=('pages', 'mean')
).reset_index().dropna()

muito_consistentes = (stats['desvio_nota'] < 0.1).sum()
moderados = ((stats['desvio_nota'] >= 0.1) & (stats['desvio_nota'] <= 0.3)).sum()
muito_irregulares = (stats['desvio_nota'] > 0.3).sum()

print(f'Total de autores analisados: {len(stats)}')
print(f'Desvio padrão médio:         {stats["desvio_nota"].mean():.3f}')
print(f'Desvio padrão mediano:       {stats["desvio_nota"].median():.3f}')
print()
print(f'Muito consistentes (desvio < 0.1):   {muito_consistentes} autores ({muito_consistentes/len(stats)*100:.1f}%)')
print(f'Moderados (desvio entre 0.1 e 0.3):  {moderados} autores ({moderados/len(stats)*100:.1f}%)')
print(f'Muito irregulares (desvio > 0.3):    {muito_irregulares} autores ({muito_irregulares/len(stats)*100:.1f}%)')

Total de autores analisados: 4978
Desvio padrão médio:         0.176
Desvio padrão mediano:       0.153

Muito consistentes (desvio < 0.1):   1318 autores (26.5%)
Moderados (desvio entre 0.1 e 0.3):  3066 autores (61.6%)
Muito irregulares (desvio > 0.3):    594 autores (11.9%)


## 1. Como a Consistência se Distribui entre os Autores?

O primeiro passo é entender se a inconsistência é um fenômeno raro ou comum no catálogo. Classificamos os autores em três grupos com base nos seguintes limites:

- **Desvio < 0.1** → autor **muito consistente**: qualidade quase constante entre todas as obras
- **Desvio entre 0.1 e 0.3** → autor **moderado**: variação aceitável
- **Desvio > 0.3** → autor **muito irregular**: obras com qualidade muito variada

In [2]:
fig = px.histogram(
    stats,
    x='desvio_nota',
    title='Distribuição da Consistência Autoral<br><sup>Desvio padrão das notas por autor: 4.527 autores com mínimo de 3 obras</sup>',
    labels={'desvio_nota': 'Desvio Padrão das Notas', 'count': 'Número de Autores'},
    color_discrete_sequence=['#C71585'],
    template='plotly_white'
)

fig.update_traces(xbins=dict(start=0, end=0.8, size=0.05))

fig.add_vrect(x0=0, x1=0.1, fillcolor='green', opacity=0.08, line_width=0)
fig.add_vrect(x0=0.1, x1=0.3, fillcolor='orange', opacity=0.08, line_width=0)
fig.add_vrect(x0=0.3, x1=0.8, fillcolor='red', opacity=0.08, line_width=0)

fig.add_annotation(x=0.05, y=1750, text='Consistente', showarrow=False,
                   font=dict(color='green', size=12))
fig.add_annotation(x=0.2, y=1750, text='Moderado', showarrow=False,
                   font=dict(color='orange', size=12))
fig.add_annotation(x=0.55, y=1750, text='Irregular', showarrow=False,
                   font=dict(color='red', size=12))

fig.add_vline(x=stats['desvio_nota'].mean(), line_dash='dash', line_color='#4B0082',
              annotation_text=f'Média: {stats["desvio_nota"].mean():.3f}',
              annotation_position='bottom right')

fig.update_layout(
    xaxis_title='Desvio Padrão das Notas',
    yaxis_title='Número de Autores',
    xaxis_range=[0, 0.8]
)
fig.show()

A distribuição revela que a maioria dos autores (57.8%) se encontra em uma faixa moderada de consistência, com desvio entre 0.1 e 0.3. Um grupo expressivo de 26.6% é muito consistente, enquanto 15.6% apresenta variação extremamente alta entre suas obras.

Esse resultado já indica que tratar todos os autores da mesma forma em um sistema de recomendação seria uma simplificação inadequada, perfis tão distintos exigem estratégias diferentes.

## 2. Os Autores Mais Consistentes do Catálogo

Identificamos os 15 autores com menor variação de qualidade entre suas obras. A cor de cada barra representa a nota média do autor, permitindo distinguir autores que são consistentemente bons daqueles que são consistentemente mediocres.

In [3]:
mais_consistentes = stats[stats['media_nota'] > 0].nsmallest(15, 'desvio_nota')

fig = px.scatter(
    mais_consistentes,
    x='media_nota',
    y='desvio_nota',
    color='media_nota',
    color_continuous_scale='Viridis',
    hover_name='author',
    hover_data={'qtd_livros': True, 'media_nota': ':.2f', 'desvio_nota': ':.4f'},
    size='qtd_livros',
    title='Top 15 Autores Mais Consistentes<br><sup>Passe o mouse para ver o autor: quanto menor o desvio, mais consistente</sup>',
    labels={'desvio_nota': 'Desvio Padrão', 'media_nota': 'Nota Média'},
    template='plotly_white'
)

fig.update_layout(showlegend=False, coloraxis_showscale=False)
fig.show()

Autores com desvio padrão próximo de zero combinam altíssima previsibilidade com qualidade estável. Para um sistema de recomendação, esses autores representam apostas seguras: qualquer obra deles tende a satisfazer o leitor que aprecia seu estilo, independentemente do título específico escolhido.

## 3. Os Autores Mais Irregulares do Catálogo

Em contraste, identificamos os 15 autores com maior variação. Aqui surge um fenômeno interessante: alguns autores irregulares possuem nota média alta, o que significa que têm obras excelentes, mas também obras significativamente abaixo do seu próprio padrão.

In [4]:
mais_irregulares = stats[stats['media_nota'] > 0].nlargest(15, 'desvio_nota')

fig = px.scatter(
    mais_irregulares,
    x='media_nota',
    y='desvio_nota',
    color='media_nota',
    color_continuous_scale='Viridis',
    hover_name='author',
    hover_data={'qtd_livros': True, 'media_nota': ':.2f', 'desvio_nota': ':.4f'},
    size='qtd_livros',
    title='Top 15 Autores Mais Irregulares<br><sup>Passe o mouse para ver o autor: tamanho indica quantidade de obras</sup>',
    labels={'desvio_nota': 'Desvio Padrão', 'media_nota': 'Nota Média'},
    template='plotly_white'
)

fig.update_layout(showlegend=False, coloraxis_showscale=False)
fig.update_layout(
    showlegend=False,
    coloraxis_showscale=False,
    xaxis_range=[1, 3],
    yaxis_range=[2, 3]
)
fig.show()

Autores com reputação consolidada mas produção heterogênea exigem atenção especial. Recomendar esses autores de forma genérica seria insuficiente — a obra específica faz toda a diferença na experiência do leitor. Isso demonstra que o título importa tanto quanto o autor em uma recomendação de qualidade.

## 4. Escrever Mais Torna um Autor Mais Irregular?

Uma hipótese natural seria que autores com mais obras no catálogo tendem a ser mais inconsistentes — afinal, com mais livros, maiores as chances de variação. Investigamos se essa relação realmente existe nos dados.

In [5]:
correlacao = stats['qtd_livros'].corr(stats['desvio_nota'])

def faixa(n):
    if n == 3: return '3 livros'
    elif n <= 5: return '4–5 livros'
    elif n <= 9: return '6–9 livros'
    else: return '10+ livros'

def perfil(d):
    if d < 0.1: return 'Consistente'
    elif d <= 0.3: return 'Moderado'
    else: return 'Irregular'

ordem_faixas = ['3 livros', '4–5 livros', '6–9 livros', '10+ livros']
ordem_perfil = ['Consistente', 'Moderado', 'Irregular']

stats['faixa'] = stats['qtd_livros'].apply(faixa)
stats['perfil'] = stats['desvio_nota'].apply(perfil)

resultado = stats.groupby(['faixa', 'perfil']).size().reset_index(name='qtd')

fig = px.bar(
    resultado,
    x='faixa',
    y='qtd',
    color='perfil',
    barmode='group',
    category_orders={'faixa': ordem_faixas, 'perfil': ordem_perfil},
    color_discrete_map={
        'Consistente': '#0C9C0C',
        'Moderado':    '#FFE433',
        'Irregular':   '#E50000'
    },
    title='Perfil de Consistência por Volume de Obras<br><sup>Quantidade de autores consistentes, moderados e irregulares em cada faixa</sup>',
    labels={
        'faixa': 'Faixa de Quantidade de Obras',
        'qtd':   'Número de Autores',
        'perfil': 'Perfil'
    },
    template='plotly_white'
)

fig.update_layout(legend_title_text='Perfil de Consistência')
fig.show()

print(f'Correlação de Pearson entre quantidade de livros e desvio padrão: {correlacao:.3f}')

Correlação de Pearson entre quantidade de livros e desvio padrão: -0.006


O gráfico revela um padrão consistente entre todas as faixas: em qualquer volume de produção, a maioria dos autores é moderada, seguida pelos consistentes e, por último, pelos irregulares. Essa proporção se mantém praticamente estável independentemente de o autor ter 3 ou 10+ livros no catálogo, confirmando visualmente a correlação fraca de 0.003. Isso significa que a quantidade de obras não é um bom preditor de consistência — a consistência parece ser uma característica intrínseca do estilo criativo de cada autor, e não uma consequência do volume de produção.

## Considerações Finais

A análise da consistência autoral revelou três achados centrais que têm impacto direto no design do sistema de recomendação do Booklog:

**A inconsistência é mais prevalente do que no dataset anterior.** Com desvio médio de 0.212 e 15.6% dos autores apresentando variação extrema (ante 9.4% antes), fica ainda mais claro que recomendar um autor não equivale a garantir uma boa experiência de leitura. A obra específica importa, e isso é uma limitação real de sistemas que recomendam apenas com base em popularidade ou nome do autor.

**Consistência e qualidade são dimensões independentes.** Autores com nota média alta mas desvio elevado possuem obras excepcionais e obras abaixo do seu próprio padrão. Um sistema que usa apenas a média do autor para sugerir qualquer obra introduz um risco real de frustrar o leitor. Já para autores com desvio próximo de zero, qualquer obra é uma aposta segura.

**Volume de produção não determina consistência.** A correlação de 0.003 desfaz a hipótese intuitiva de que autores prolíficos são mais irregulares. Isso significa que o número de livros de um autor no catálogo não deve ser utilizado como proxy de confiabilidade em um algoritmo de recomendação — cada autor precisa ser avaliado individualmente pelo seu desvio padrão.

Em conjunto, esses achados sugerem que o desvio padrão das notas por autor é uma feature relevante a ser incorporada no modelo de Machine Learning do Booklog, permitindo que o sistema adote estratégias de recomendação distintas para autores consistentes e irregulares.